# 08 - Imputación de valores ausentes
Toma el dataset ajustado (17 embalses experimentales, 2006-2023) que produce el notebook de ajuste y completa los huecos cortos de la variable objetivo y de las predictoras continuas, dejándolo listo para la ingeniería de variables.

## 1. Configuración

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")

# Longitud máxima de racha admitida para imputación (días)
MAX_RACHA_IMPUTABLE = 7
# Frontera train/test: la imputación no cruza esta fecha
CORTE_TEST = pd.Timestamp("2022-01-01")

dataset = (pd.read_parquet(DIR_PROCESSED / "dataset_ajustado.parquet")
           .sort_values(["ID_SAIH", "fecha"])
           .reset_index(drop=True))

print(f"Dataset ajustado: {len(dataset):,} filas | "
      f"{dataset['ID_SAIH'].nunique()} embalses")
print(f"Periodo: {dataset['fecha'].min():%Y-%m-%d} a {dataset['fecha'].max():%Y-%m-%d}")
print(f"Columnas ({len(dataset.columns)}): {list(dataset.columns)}")

Dataset ajustado: 111,758 filas | 17 embalses
Periodo: 2006-01-01 a 2023-12-31
Columnas (20): ['fecha', 'ID_SAIH', 'volumen_hm3', 'pct_llenado', 'aportacion_m3s', 'salida_m3s', 'aemet_temp_media_c', 'aemet_temp_min_c', 'aemet_temp_max_c', 'aemet_precipitacion_mm', 'aemet_humedad_pct', 'Nombre_SAIH', 'Sistema', 'Capacidad_hm3', 'cal_amonio_mgl', 'cal_conductividad_uscm', 'cal_oxigeno_mgl', 'cal_ph', 'cal_temp_agua_c', 'cal_turbidez_ntu']


## 2. Imputación de la variable objetivo y de las predictoras continuas


In [2]:
OBJETIVO = "pct_llenado"

# Metadatos y calidad quedan fuera de la imputación
META = ["fecha", "ID_SAIH", "Nombre_SAIH", "Sistema", "Capacidad_hm3"]
CALIDAD = [c for c in dataset.columns if c.startswith("cal_")]

PREDICTORAS = [c for c in dataset.columns
               if c not in META + CALIDAD + [OBJETIVO]]

print(f"Objetivo:    {OBJETIVO}")
print(f"Predictoras a imputar ({len(PREDICTORAS)}): {PREDICTORAS}")
print(f"Calidad NO imputada ({len(CALIDAD)}): {CALIDAD}")


def imputar_bloque(df):
    """Imputa un bloque temporal de forma independiente, de modo que ningún valor
    del conjunto de test intervenga en el relleno del de entrenamiento."""
    d = df.sort_values(["ID_SAIH", "fecha"]).copy()
    # Objetivo: arrastre del último valor (nunca futuro)
    d[OBJETIVO] = d.groupby("ID_SAIH")[OBJETIVO].transform(
        lambda s: s.ffill(limit=MAX_RACHA_IMPUTABLE))
    # Predictoras continuas: arrastre del último valor observado (nunca futuro),
    # para no introducir información posterior al instante imputado
    for c in PREDICTORAS:
        d[c] = d.groupby("ID_SAIH")[c].transform(
            lambda s: s.ffill(limit=MAX_RACHA_IMPUTABLE))
    return d


antes = dataset[[OBJETIVO] + PREDICTORAS].isna().sum()

dataset_imp = (pd.concat([
    imputar_bloque(dataset[dataset["fecha"] < CORTE_TEST]),
    imputar_bloque(dataset[dataset["fecha"] >= CORTE_TEST]),
], ignore_index=True)
    .sort_values(["ID_SAIH", "fecha"])
    .reset_index(drop=True))

despues = dataset_imp[[OBJETIVO] + PREDICTORAS].isna().sum()
comp = pd.DataFrame({"antes": antes, "despues": despues})
comp["imputados"] = comp["antes"] - comp["despues"]
comp["cobertura_%"] = ((1 - comp["despues"] / len(dataset_imp)) * 100).round(3)
print()
print(comp.to_string())

Objetivo:    pct_llenado
Predictoras a imputar (8): ['volumen_hm3', 'aportacion_m3s', 'salida_m3s', 'aemet_temp_media_c', 'aemet_temp_min_c', 'aemet_temp_max_c', 'aemet_precipitacion_mm', 'aemet_humedad_pct']
Calidad NO imputada (6): ['cal_amonio_mgl', 'cal_conductividad_uscm', 'cal_oxigeno_mgl', 'cal_ph', 'cal_temp_agua_c', 'cal_turbidez_ntu']

                        antes  despues  imputados  cobertura_%
pct_llenado               346        0        346      100.000
volumen_hm3               346        0        346      100.000
aportacion_m3s            469        0        469      100.000
salida_m3s                375        3        372       99.997
aemet_temp_media_c       2622     1829        793       98.363
aemet_temp_min_c         2620     1829        791       98.363
aemet_temp_max_c         2616     1829        787       98.363
aemet_precipitacion_mm   3922     1927       1995       98.276
aemet_humedad_pct        5451     4672        779       95.820


## 3. Huecos residuales en las variables meteorológicas


In [3]:
METEO = [c for c in dataset_imp.columns if c.startswith("aemet_")]

antes_clim = dataset_imp[METEO].isna().sum()

dataset_imp["_dia"] = dataset_imp["fecha"].dt.dayofyear
train_mask = dataset_imp["fecha"] < CORTE_TEST

for c in METEO:
    if not dataset_imp[c].isna().any():
        continue
    # Media por embalse y día del año, calculada solo sobre entrenamiento
    clim = (dataset_imp[train_mask]
            .groupby(["ID_SAIH", "_dia"])[c].mean()
            .rename("_clim").reset_index())
    dataset_imp = dataset_imp.merge(clim, on=["ID_SAIH", "_dia"], how="left")
    dataset_imp[c] = dataset_imp[c].fillna(dataset_imp["_clim"])
    dataset_imp = dataset_imp.drop(columns="_clim")

dataset_imp = dataset_imp.drop(columns="_dia")

despues_clim = dataset_imp[METEO].isna().sum()
comp_clim = pd.DataFrame({"antes": antes_clim, "despues": despues_clim})
comp_clim["completados"] = comp_clim["antes"] - comp_clim["despues"]
comp_clim["cobertura_%"] = ((1 - comp_clim["despues"] / len(dataset_imp)) * 100).round(1)
print(comp_clim.to_string())

                        antes  despues  completados  cobertura_%
aemet_temp_media_c       1829        0         1829        100.0
aemet_temp_min_c         1829        0         1829        100.0
aemet_temp_max_c         1829        0         1829        100.0
aemet_precipitacion_mm   1927        0         1927        100.0
aemet_humedad_pct        4672        0         4672        100.0


# 4. Huecos calidad (entre campañas)

In [4]:
def _arrastrar_serie(s):
    """Arrastra el último valor medido hacia delante sobre la serie completa. El arrastre
    es causal (cada día toma el último valor conocido), por lo que no introduce
    información futura aunque cruce la frontera train/test: el último valor del
    entrenamiento sí está disponible al inicio del test. Solo quedan sin dato los días
    anteriores a la primera medición de la serie."""
    return s.ffill()

antes_cal = dataset_imp[CALIDAD].notna().sum()

# Trabajamos sobre un índice (ID_SAIH, fecha) para alinear sin ambigüedad posicional
_base = dataset_imp.set_index(["ID_SAIH", "fecha"]).sort_index()
for c in CALIDAD:
    _base[c] = (_base[c]
                .groupby(level="ID_SAIH", group_keys=False)
                .apply(lambda s: _arrastrar_serie(s.droplevel("ID_SAIH"))
                       .rename(c).rename_axis("fecha"))
                .values)
dataset_imp = _base.reset_index()[dataset_imp.columns]

despues_cal = dataset_imp[CALIDAD].notna().sum()
comp_cal = pd.DataFrame({"antes": antes_cal, "despues": despues_cal})
comp_cal["arrastrados"] = comp_cal["despues"] - comp_cal["antes"]
comp_cal["cobertura_%"] = (dataset_imp[CALIDAD].notna().mean() * 100).round(1)
print("Arrastre de calidad:")
print(comp_cal.to_string())

Arrastre de calidad:
                         antes  despues  arrastrados  cobertura_%
cal_amonio_mgl           92024   111758        19734        100.0
cal_conductividad_uscm  103767   111758         7991        100.0
cal_oxigeno_mgl         103000   111758         8758        100.0
cal_ph                  103754   111758         8004        100.0
cal_temp_agua_c         104080   111758         7678        100.0
cal_turbidez_ntu        100107   111758        11651        100.0


## 4. Comprobación: cobertura resultante

In [5]:
print("\nCobertura del objetivo y predictoras tras imputación:")
for c in [OBJETIVO] + PREDICTORAS + CALIDAD:
    print(f"  {c:<26} {dataset_imp[c].isna().sum():5.0f} -> {dataset_imp[c].notna().mean()*100:5.5f}%")


Cobertura del objetivo y predictoras tras imputación:
  pct_llenado                    0 -> 100.00000%
  volumen_hm3                    0 -> 100.00000%
  aportacion_m3s                 0 -> 100.00000%
  salida_m3s                     3 -> 99.99732%
  aemet_temp_media_c             0 -> 100.00000%
  aemet_temp_min_c               0 -> 100.00000%
  aemet_temp_max_c               0 -> 100.00000%
  aemet_precipitacion_mm         0 -> 100.00000%
  aemet_humedad_pct              0 -> 100.00000%
  cal_amonio_mgl                 0 -> 100.00000%
  cal_conductividad_uscm         0 -> 100.00000%
  cal_oxigeno_mgl                0 -> 100.00000%
  cal_ph                         0 -> 100.00000%
  cal_temp_agua_c                0 -> 100.00000%
  cal_turbidez_ntu               0 -> 100.00000%


## 5. Guardado

In [6]:
dataset_imp.to_parquet(DIR_PROCESSED / "dataset_imputado.parquet", index=False)

print(f"Guardado: {len(dataset_imp):,} filas x {dataset_imp.shape[1]} columnas")
print(f"Embalses: {dataset_imp['ID_SAIH'].nunique()}")
print(f"Periodo: {dataset_imp['fecha'].min():%Y-%m-%d} a {dataset_imp['fecha'].max():%Y-%m-%d}")

Guardado: 111,758 filas x 20 columnas
Embalses: 17
Periodo: 2006-01-01 a 2023-12-31
